<a href="https://colab.research.google.com/github/aadhilahamath07-pixel/AlCu-Laser-Welding-ML/blob/main/notebooks/02_conventional_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/aadhilahamath07-pixel/AlCu-Laser-Welding-ML.git

%cd /content/AlCu-Laser-Welding-ML

Cloning into 'AlCu-Laser-Welding-ML'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 46 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 35.34 KiB | 8.83 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/AlCu-Laser-Welding-ML


# 02 — Conventional laser-welding data audit and cleaning

This notebook works on a copy of `data/raw/Laser Welding.xlsx`. It preserves the source workbook and does not automatically delete rows, replace missing values, or convert zero measurements into missing values.

First we create a transparent audit. Cleaning decisions will be made only after reviewing the actual columns and records.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "Laser Welding.xlsx"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_FILE.exists(), f"Cannot find source workbook: {RAW_FILE}"
print("Source workbook:", RAW_FILE)
print("Output folder:", OUT_DIR)

Source workbook: /content/AlCu-Laser-Welding-ML/data/raw/Laser Welding.xlsx
Output folder: /content/AlCu-Laser-Welding-ML/data/processed


## 1. Load every sheet without changing the source

In [3]:
source_book = pd.ExcelFile(RAW_FILE)
print("Sheets:", source_book.sheet_names)
raw_sheets = {name: pd.read_excel(RAW_FILE, sheet_name=name) for name in source_book.sheet_names}
for name, frame in raw_sheets.items():
    print(f"\n--- {name}: {frame.shape[0]} rows × {frame.shape[1]} columns ---")
    display(frame.head(10))

Sheets: ['Data']

--- Data: 27 rows × 5 columns ---


,Laser Power [W],Welding Speed [mm/s],Weld bead width [mm],Interface width [mm],Weld bead penetration depth [mm]
0,500,200,1.25,0.48,0.27
1,500,200,1.15,0.27,0.30
2,500,200,1.28,0.35,0.10
3,500,450,0.00,0.00,0.00
4,500,450,0.00,0.00,0.00
5,500,450,0.00,0.00,0.00
6,500,700,0.00,0.00,0.00
7,500,700,0.00,0.00,0.00
8,500,700,0.00,0.00,0.00
9,750,200,0.80,0.22,0.30


## 2. Audit column names, data types, missing values and duplicates

This report is diagnostic. Duplicate rows are flagged for review, not removed.

In [4]:
audit_rows = []
for sheet_name, frame in raw_sheets.items():
    for column in frame.columns:
        audit_rows.append({
            "sheet": sheet_name,
            "column": str(column),
            "dtype": str(frame[column].dtype),
            "non_missing": int(frame[column].notna().sum()),
            "missing": int(frame[column].isna().sum()),
            "unique_non_missing": int(frame[column].nunique(dropna=True))
        })
    print(f"\n{sheet_name}: exact duplicate rows = {int(frame.duplicated().sum())}")
column_audit = pd.DataFrame(audit_rows)
display(column_audit)


Data: exact duplicate rows = 4


,sheet,column,dtype,non_missing,missing,unique_non_missing
0,Data,Laser Power [W],int64,27,0,3
1,Data,Welding Speed [mm/s],int64,27,0,3
2,Data,Weld bead width [mm],float64,27,0,19
3,Data,Interface width [mm],float64,27,0,15
4,Data,Weld bead penetration depth [mm],float64,27,0,11


## 3. Flag zero-valued numeric measurements for investigation

A zero can mean a genuine measured zero, no weld, lack of fusion, or a placeholder. We must use the source's experimental meaning before deciding how to treat it.

In [5]:
zero_reports = []
for sheet_name, frame in raw_sheets.items():
    numeric = frame.select_dtypes(include="number")
    for column in numeric.columns:
        count = int((numeric[column] == 0).sum())
        if count:
            zero_reports.append({"sheet": sheet_name, "column": str(column), "zero_count": count})
zero_audit = pd.DataFrame(zero_reports, columns=["sheet", "column", "zero_count"])
display(zero_audit if not zero_audit.empty else pd.DataFrame({"result": ["No numeric zero values found"]}))

,sheet,column,zero_count
0,Data,Weld bead width [mm],6
1,Data,Interface width [mm],6
2,Data,Weld bead penetration depth [mm],6


## 4. Create an untouched copy for the next cleaning stage

This export preserves all rows and values. It is a checkpoint, not yet the final cleaned dataset.

In [6]:
checkpoint = OUT_DIR / "conventional_laser_welding_audit_copy.xlsx"
with pd.ExcelWriter(checkpoint, engine="openpyxl") as writer:
    for sheet_name, frame in raw_sheets.items():
        frame.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)
    column_audit.to_excel(writer, sheet_name="Column_Audit", index=False)
    zero_audit.to_excel(writer, sheet_name="Zero_Value_Audit", index=False)
print("Saved audit copy:", checkpoint)
print("Original file remains unchanged:", RAW_FILE.exists())

Saved audit copy: /content/AlCu-Laser-Welding-ML/data/processed/conventional_laser_welding_audit_copy.xlsx
Original file remains unchanged: True


## 5. Stop and review before applying transformations

Do not drop rows or fill missing values yet. Confirm the meaning of zero-valued outputs and identify which columns are inputs, outputs, specimen identifiers, and repeat numbers. The next cleaning revision will standardize the confirmed fields and document each transformation.

In [8]:

import pandas as pd

file_path = "data/processed/conventional_laser_welding_audit_copy.xlsx"

xls = pd.ExcelFile(file_path)

print("Sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    print("\n==========", sheet, "==========")
    df = pd.read_excel(file_path, sheet_name=sheet)
    display(df)


Sheets: ['Data', 'Column_Audit', 'Zero_Value_Audit']

========== Data ==========


,Laser Power [W],Welding Speed [mm/s],Weld bead width [mm],Interface width [mm],Weld bead penetration depth [mm]
0,500,200,1.25,0.48,0.27
1,500,200,1.15,0.27,0.30
2,500,200,1.28,0.35,0.10
3,500,450,0.00,0.00,0.00
4,500,450,0.00,0.00,0.00
5,500,450,0.00,0.00,0.00
6,500,700,0.00,0.00,0.00
7,500,700,0.00,0.00,0.00
8,500,700,0.00,0.00,0.00
9,750,200,0.80,0.22,0.30



========== Column_Audit ==========


,sheet,column,dtype,non_missing,missing,unique_non_missing
0,Data,Laser Power [W],int64,27,0,3
1,Data,Welding Speed [mm/s],int64,27,0,3
2,Data,Weld bead width [mm],float64,27,0,19
3,Data,Interface width [mm],float64,27,0,15
4,Data,Weld bead penetration depth [mm],float64,27,0,11



========== Zero_Value_Audit ==========


,sheet,column,zero_count
0,Data,Weld bead width [mm],6
1,Data,Interface width [mm],6
2,Data,Weld bead penetration depth [mm],6
